# I. Giới thiệu

Dự đoán sớm sinh viên có nguy cơ trượt hoặc bỏ học cho phép can thiệp kịp thời và đã được
nghiên cứu rộng rãi trên dữ liệu học tập trực tuyến [1]. Đồ án này xây dựng một pipeline học
máy *khả diễn giải, có nhận thức về thời gian*, dự đoán nguy cơ tại sáu mốc tiến độ
(10%, 20%, 40%, 60%, 80%, 100%) của môn học trên bộ Open University Learning Analytics Dataset
(OULAD) [2]. Trước khi mô hình hoá, một bước khảo sát dữ liệu vững chắc là cần thiết: hiểu cấu
trúc các bảng và xác nhận chúng kết nối được với nhau.

Báo cáo này (hạng mục STT 9) trình bày khảo sát đó. Đóng góp gồm: (i) thống kê kích thước và
kiểu dữ liệu của bảy bảng OULAD; (ii) xác nhận khoá tổng hợp `id_student × code_module ×
code_presentation`; (iii) kiểm tra khả năng kết nối qua ba phép kiểm; và (iv) một sơ đồ quan hệ
thực thể (ER). Theo "mười quy tắc cho phân tích trên Jupyter notebook" [3], tài liệu này được
thiết kế để vừa **đọc** (phần tường thuật), vừa **chạy** (mã mô-đun hoá, phụ thuộc tường minh),
vừa **khám phá** (bảng và sơ đồ sinh trực tiếp từ dữ liệu).

# II. Bộ dữ liệu OULAD

OULAD [2] là bộ dữ liệu mở gồm 32.593 sinh viên trên 22 môn–kỳ, với khoảng 10,6 triệu lượt
tương tác trên môi trường học trực tuyến (VLE). Dữ liệu được tổ chức thành **bảy bảng quan hệ**
ở các cấp độ hạt (*grain*) khác nhau: `courses` là bảng chiều ở cấp môn–kỳ; `studentInfo` và
`studentRegistration` ở cấp sinh viên–môn–kỳ; `assessments` và `vle` định nghĩa thực thể bài
đánh giá và tài nguyên học liệu; còn `studentAssessment` và `studentVle` là hai bảng **sự kiện**
ghi nhận hành vi theo thời gian. Trường `final_result` của `studentInfo` là nguồn nhãn cho bài
toán phân loại nhị phân *at-risk / not-at-risk*. Khảo sát dưới đây xác nhận bảy bảng này có thể
được hợp nhất một cách nhất quán.

# III. Phương pháp khảo sát

Quy trình gồm bốn bước: (1) đọc bảy bảng và thống kê `.shape`, `.dtypes`; (2) xác định khoá
tổng hợp; (3) kiểm tra hiện diện khoá, tính duy nhất và toàn vẹn tham chiếu; (4) dựng sơ đồ ER.
Để bảo đảm khả tái lập, ô cấu hình khai báo đường dẫn dữ liệu, danh mục bảng, khoá tổng hợp và
một tiện ích trình bày bảng dùng chung; ô nạp dữ liệu đóng gói toàn bộ logic đọc/sinh dữ liệu
thành các hàm độc lập (*modular*).

In [ ]:
# Phụ thuộc tường minh của pipeline khảo sát.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Thư mục chứa bảy tệp CSV gốc của OULAD (đổi đường dẫn nếu đặt nơi khác).
DATA_DIR = Path("data/raw")

# Bảy bảng OULAD theo đặc tả Kuzilek, Hlosta & Zdrahal (2017) [2] và tên tệp tương ứng.
TABLE_FILES = {
    "courses":             "courses.csv",
    "studentInfo":         "studentInfo.csv",
    "studentRegistration": "studentRegistration.csv",
    "assessments":         "assessments.csv",
    "studentAssessment":   "studentAssessment.csv",
    "vle":                 "vle.csv",
    "studentVle":          "studentVle.csv",
}

# Khoá tổng hợp ở cấp sinh viên–môn–kỳ.
COMPOSITE_KEY = ["id_student", "code_module", "code_presentation"]

# --- Tiện ích trình bày: bảng ĐƠN SẮC, đọc được ở cả nền sáng và nền tối ---
# Không đặt màu chữ/nền cố định -> chữ kế thừa màu giao diện (sáng hoặc tối). Kẻ và
# dòng sọc dùng sắc xám bán trong suốt (rgba) cùng currentColor nên luôn rõ ở hai chế độ.
def style_table(df: pd.DataFrame, right=None, center=None, caption: str | None = None):
    """Trả về Styler đơn sắc: tiêu đề đậm có kẻ chân, dòng sọc xám nhạt, chú thích dưới bảng."""
    right, center = right or [], center or []
    base = [
        {"selector": "table",
         "props": [("border-collapse", "collapse"),
                   ("font-family", "Helvetica, Arial, sans-serif"), ("font-size", "13px")]},
        {"selector": "th",
         "props": [("font-weight", "700"), ("text-align", "left"), ("padding", "6px 12px"),
                   ("border", "1px solid rgba(128,128,128,0.45)"),
                   ("border-bottom", "2px solid currentColor")]},
        {"selector": "td",
         "props": [("padding", "5px 12px"), ("border", "1px solid rgba(128,128,128,0.35)")]},
        {"selector": "tbody tr:nth-child(even)",
         "props": [("background-color", "rgba(128,128,128,0.10)")]},
        {"selector": "caption",
         "props": [("caption-side", "bottom"), ("padding", "6px 2px"),
                   ("font-style", "italic"), ("opacity", "0.7"), ("font-size", "12px")]},
    ]
    sty = df.style.hide(axis="index").set_table_styles(base)
    for col in right:
        sty = sty.set_properties(subset=[col], **{"text-align": "right"})
    for col in center:
        sty = sty.set_properties(subset=[col], **{"text-align": "center"})
    if caption:
        sty = sty.set_caption(caption)
    return sty


print(f"pandas {pd.__version__}  ·  numpy {np.__version__}  ·  Python {sys.version.split()[0]}")
print("DATA_DIR        :", DATA_DIR.resolve())
print("Khoá tổng hợp   :", " × ".join(COMPOSITE_KEY))

In [ ]:
def build_synthetic_oulad(seed: int = 7) -> dict[str, pd.DataFrame]:
    """Sinh một mẫu OULAD tổng hợp: đúng tên cột, đúng kiểu, nhất quán tham chiếu.

    Cho phép notebook CHẠY được khi chưa tải dữ liệu thật; số bản ghi cố tình nhỏ. Tính
    nhất quán tham chiếu (mọi ``id_assessment``/``id_site`` con đều có ở bảng cha) được bảo
    đảm để các phép kiểm tra kết nối ở Mục IV minh hoạ đúng kết quả.
    """
    rng = np.random.default_rng(seed)
    modules, presentations = ["AAA", "BBB", "CCC"], ["2013J", "2014J"]

    courses = pd.DataFrame(
        [(m, p, int(rng.integers(234, 270))) for m in modules for p in presentations],
        columns=["code_module", "code_presentation", "module_presentation_length"],
    )

    info_rows, reg_rows, enrolments = [], [], []
    final_levels = ["Pass", "Fail", "Withdrawn", "Distinction"]
    for sid in range(1, 61):                       # 60 sinh viên
        picks = rng.choice(len(courses), size=int(rng.integers(1, 3)), replace=False)
        for idx in picks:                          # mỗi SV học 1–2 môn–kỳ
            m, p = courses.loc[idx, "code_module"], courses.loc[idx, "code_presentation"]
            enrolments.append((sid, m, p))
            info_rows.append((sid, m, p, rng.choice(["M", "F"]),
                              rng.choice(["East", "West", "North", "South"]),
                              rng.choice(["A Level", "HE", "Lower Than A Level"]),
                              rng.choice(["0-10%", "30-40%", "60-70%", "90-100%"]),
                              rng.choice(["0-35", "35-55", "55<="]),
                              int(rng.integers(0, 3)), int(rng.integers(30, 240)),
                              rng.choice(["N", "Y"]), rng.choice(final_levels)))
            reg_rows.append((sid, m, p, int(rng.integers(-30, 0)),
                             np.nan if rng.random() < 0.7 else int(rng.integers(20, 240))))
    student_info = pd.DataFrame(info_rows, columns=[
        "id_student", "code_module", "code_presentation", "gender", "region",
        "highest_education", "imd_band", "age_band", "num_of_prev_attempts",
        "studied_credits", "disability", "final_result"])
    student_registration = pd.DataFrame(reg_rows, columns=[
        "id_student", "code_module", "code_presentation",
        "date_registration", "date_unregistration"])

    assess_rows, site_rows, aid, sid_site = [], [], 1000, 5000
    assess_by_course, sites_by_course = {}, {}
    for _, c in courses.iterrows():
        key = (c["code_module"], c["code_presentation"])
        assess_by_course[key] = []
        for atype, day, w in [("TMA", 60, 20.0), ("CMA", 130, 10.0),
                              ("Exam", c["module_presentation_length"], 70.0)]:
            assess_rows.append((aid, c["code_module"], c["code_presentation"], atype, float(day), w))
            assess_by_course[key].append(aid); aid += 1
        sites_by_course[key] = []
        for _ in range(5):                          # 5 tài nguyên VLE mỗi môn–kỳ
            site_rows.append((sid_site, c["code_module"], c["code_presentation"],
                              rng.choice(["resource", "url", "quiz", "forumng"]),
                              float(rng.integers(0, 5)), float(rng.integers(20, 30))))
            sites_by_course[key].append(sid_site); sid_site += 1
    assessments = pd.DataFrame(assess_rows, columns=[
        "id_assessment", "code_module", "code_presentation", "assessment_type", "date", "weight"])
    vle = pd.DataFrame(site_rows, columns=[
        "id_site", "code_module", "code_presentation", "activity_type", "week_from", "week_to"])

    sa_rows, sv_rows = [], []
    for sid, m, p in enrolments:
        key = (m, p)
        for a in assess_by_course[key]:
            if rng.random() < 0.8:                  # nộp ~80% số bài
                sa_rows.append((a, sid, int(rng.integers(40, 230)), 0,
                                float(round(rng.uniform(40, 100), 1))))
        for s in sites_by_course[key]:
            for _ in range(int(rng.integers(0, 6))):   # số lượt click
                sv_rows.append((m, p, sid, s, int(rng.integers(-20, 240)), int(rng.integers(1, 12))))
    student_assessment = pd.DataFrame(sa_rows, columns=[
        "id_assessment", "id_student", "date_submitted", "is_banked", "score"])
    student_vle = pd.DataFrame(sv_rows, columns=[
        "code_module", "code_presentation", "id_student", "id_site", "date", "sum_click"])

    return {"courses": courses, "studentInfo": student_info,
            "studentRegistration": student_registration, "assessments": assessments,
            "studentAssessment": student_assessment, "vle": vle, "studentVle": student_vle}


def load_tables() -> tuple[dict[str, pd.DataFrame], bool]:
    """Đọc dữ liệu thật nếu đủ bảy tệp trong DATA_DIR; nếu thiếu, dùng mẫu tổng hợp."""
    have_real = all((DATA_DIR / fname).exists() for fname in TABLE_FILES.values())
    if have_real:
        data = {name: pd.read_csv(DATA_DIR / fname) for name, fname in TABLE_FILES.items()}
    else:
        data = build_synthetic_oulad()
    return data, have_real


tables, USING_REAL_DATA = load_tables()
print("Chế độ dữ liệu:",
      "DỮ LIỆU THẬT (data/raw/)" if USING_REAL_DATA
      else "DỮ LIỆU MẪU TỔNG HỢP — chưa có OULAD; thay tệp thật rồi Run All để cập nhật.")

# IV. Kết quả và thảo luận

## A. Kích thước và kiểu dữ liệu

Bảng I tổng hợp số dòng/cột của bảy bảng; Bảng II là **từ điển dữ liệu** liệt kê kiểu dữ liệu
của mọi cột, làm tham chiếu cho các bước tiền xử lý sau.

In [ ]:
# BẢNG I — kích thước bảy bảng (.shape).
shape_summary = pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in tables.items()],
    columns=["Bảng", "Số dòng", "Số cột"],
)
style_table(shape_summary, right=["Số dòng", "Số cột"],
            caption="BẢNG I. Kích thước bảy bảng OULAD đã nạp.")

In [ ]:
# BẢNG II — từ điển dữ liệu: kiểu dữ liệu (.dtypes) của mọi cột.
records = []
for name, df in tables.items():
    for j, (col, dtype) in enumerate(zip(df.columns, df.dtypes.astype(str))):
        records.append((name if j == 0 else "", col, dtype))
data_dictionary = pd.DataFrame(records, columns=["Bảng", "Cột", "Kiểu dữ liệu"])
style_table(data_dictionary,
            caption="BẢNG II. Từ điển dữ liệu — kiểu dữ liệu từng cột của bảy bảng.")

Bảy bảng nằm ở ba cấp độ hạt; hai bảng sự kiện (`studentAssessment`,
`studentVle`) chiếm phần lớn số dòng, đúng như kỳ vọng đối với dữ liệu hành vi theo thời gian.

## B. Khoá tổng hợp và khả năng kết nối

Ba phép kiểm xác nhận bảy bảng có thể hợp nhất: (1) hiện diện của khoá (Bảng III); (2) tính duy
nhất của khoá tổng hợp trong `studentInfo`/`studentRegistration`; và (3) toàn vẹn tham chiếu —
mọi `id_assessment` trong `studentAssessment` phải tồn tại trong `assessments`, mọi `id_site`
trong `studentVle` phải tồn tại trong `vle` (Bảng IV).

In [ ]:
# BẢNG III — ma trận hiện diện của ba cột khoá trong bảy bảng.
key_cols = ["id_student", "code_module", "code_presentation"]
presence = pd.DataFrame(
    {col: [("✓" if col in tables[name].columns else "—") for name in TABLE_FILES]
     for col in key_cols},
    index=list(TABLE_FILES),
).reset_index(names="Bảng")

# Hiển thị đơn sắc: ✓ = có khoá, — = không có (căn giữa, không dùng màu).
style_table(presence, center=key_cols,
            caption="BẢNG III. Hiện diện của khoá tổng hợp trong từng bảng.")

In [ ]:
# BẢNG IV — kiểm tra tính duy nhất và toàn vẹn tham chiếu.
def _verdict(passed: bool) -> str:
    return "Đạt" if passed else "KHÔNG đạt"

rows = []
for name in ["studentInfo", "studentRegistration"]:
    n_dup = int(tables[name].duplicated(subset=COMPOSITE_KEY).sum())
    rows.append(("Tính duy nhất của khoá", name, f"{n_dup} dòng trùng", _verdict(n_dup == 0)))

orphan_sa = int((~tables["studentAssessment"]["id_assessment"]
                 .isin(tables["assessments"]["id_assessment"])).sum())
orphan_sv = int((~tables["studentVle"]["id_site"]
                 .isin(tables["vle"]["id_site"])).sum())
rows.append(("Toàn vẹn tham chiếu", "studentAssessment → assessments",
             f"{orphan_sa} bản ghi mồ côi", _verdict(orphan_sa == 0)))
rows.append(("Toàn vẹn tham chiếu", "studentVle → vle",
             f"{orphan_sv} bản ghi mồ côi", _verdict(orphan_sv == 0)))

checks = pd.DataFrame(rows, columns=["Phép kiểm tra", "Đối tượng", "Kết quả", "Kết luận"])

# Cột "Kết luận" dùng chữ (Đạt / KHÔNG đạt) thay cho màu, đọc được ở cả nền sáng/tối.
style_table(checks, center=["Kết luận"],
            caption="BẢNG IV. Kết quả kiểm tra khả năng kết nối.")

Khoá tổng hợp hiện diện đầy đủ ở các bảng cấp sinh viên–môn–kỳ và là duy nhất; không
tồn tại bản ghi mồ côi. Bảy bảng do đó có thể hợp nhất an toàn thành *master table*.

## C. Sơ đồ quan hệ thực thể

Hình 1 tổng hợp lược đồ: khoá chính được đánh dấu `★`, các cạnh thể hiện quan hệ một–nhiều.

In [ ]:
from graphviz import Digraph

# Sơ đồ ĐƠN SẮC (thang xám) trên nền trắng — rõ ràng ở cả giao diện sáng và tối.
HDR_BG, KEY_BG, ATTR_BG = "#3A3A3A", "#ECECEC", "#FFFFFF"
GRAIN_BG, EDGE, TXT = "#F5F5F5", "#707070", "#1A1A1A"

er = Digraph("OULAD_ER", format="png")
er.attr(rankdir="LR", splines="spline", nodesep="0.5", ranksep="0.9",
        bgcolor="white", fontname="Helvetica")
er.attr("node", shape="plaintext", fontname="Helvetica")
er.attr("edge", color=EDGE, fontname="Helvetica", fontsize="10", penwidth="1.4")


def er_table(name: str, grain: str, rows: list[tuple[str, bool]]) -> str:
    """Dựng một node-bảng dạng HTML cho Graphviz: tiêu đề + các cột (khoá in đậm)."""
    body = (
        f'<<table border="0" cellborder="1" cellspacing="0" cellpadding="5">'
        f'<tr><td bgcolor="{HDR_BG}"><font color="white"><b>{name}</b></font></td></tr>'
        f'<tr><td bgcolor="{GRAIN_BG}"><font color="{TXT}" point-size="9"><i>{grain}</i></font></td></tr>'
    )
    for col, is_key in rows:
        mark = f"<b>{col}</b>" if is_key else col
        shade = KEY_BG if is_key else ATTR_BG
        body += (f'<tr><td bgcolor="{shade}" align="left">'
                 f'<font color="{TXT}" point-size="10">{mark}</font></td></tr>')
    return body + "</table>>"


er.node("courses", er_table("courses", "1 dòng / môn-kỳ", [
    ("code_module ★", True), ("code_presentation ★", True),
    ("module_presentation_length", False)]))
er.node("studentInfo", er_table("studentInfo", "1 dòng / SV-môn-kỳ", [
    ("id_student ★", True), ("code_module ★", True), ("code_presentation ★", True),
    ("gender, region, age_band", False), ("highest_education, imd_band", False),
    ("studied_credits, disability", False), ("final_result  →  nhãn", False)]))
er.node("studentRegistration", er_table("studentRegistration", "1 dòng / SV-môn-kỳ", [
    ("id_student ★", True), ("code_module ★", True), ("code_presentation ★", True),
    ("date_registration", False), ("date_unregistration", False)]))
er.node("assessments", er_table("assessments", "1 dòng / bài đánh giá", [
    ("id_assessment ★", True), ("code_module, code_presentation", False),
    ("assessment_type, date, weight", False)]))
er.node("studentAssessment", er_table("studentAssessment", "1 dòng / SV-bài", [
    ("id_assessment ★", True), ("id_student ★", True),
    ("date_submitted, is_banked, score", False)]))
er.node("vle", er_table("vle", "1 dòng / tài nguyên VLE", [
    ("id_site ★", True), ("code_module, code_presentation", False),
    ("activity_type, week_from, week_to", False)]))
er.node("studentVle", er_table("studentVle", "1 dòng / lượt click (~10 triệu)", [
    ("id_student ★", True), ("id_site ★", True),
    ("code_module, code_presentation", False), ("date, sum_click", False)]))

# Quan hệ 1..N: đầu '1' tại bảng cha, 'N' tại bảng con.
for a, b, ta, hb in [
    ("courses", "studentInfo", "1", "N"), ("courses", "assessments", "1", "N"),
    ("courses", "vle", "1", "N"), ("studentInfo", "studentRegistration", "1", "1"),
    ("studentInfo", "studentAssessment", "1", "N"), ("studentInfo", "studentVle", "1", "N"),
    ("assessments", "studentAssessment", "1", "N"), ("vle", "studentVle", "1", "N"),
]:
    er.edge(a, b, taillabel=ta, headlabel=hb, labeldistance="1.6", labelangle="0")

# Kết xuất PNG để lưu kèm và hiển thị nội tuyến.
png_bytes = er.pipe(format="png")
with open("oulad_er.png", "wb") as fh:
    fh.write(png_bytes)

from IPython.display import Image
Image(png_bytes)

*Hình 1. Sơ đồ quan hệ thực thể của bảy bảng OULAD ($\star$ = khoá chính;
$1\text{–}N$ = quan hệ một–nhiều).*

# V. Kết luận

Khảo sát xác nhận bảy bảng OULAD ở ba cấp độ hạt, với khoá tổng hợp `id_student × code_module ×
code_presentation` hiện diện đầy đủ ở các bảng cấp sinh viên–môn–kỳ, duy nhất, và không có bản
ghi mồ côi giữa bảng sự kiện và bảng cha. Sơ đồ ER cung cấp bản thiết kế cho bước hợp nhất dữ
liệu. Đối chiếu tiêu chí nghiệm thu STT 9 — khảo sát đủ bảy bảng và chứng minh khả năng kết nối
qua khoá tổng hợp — báo cáo đạt yêu cầu.

Khi chạy trên dữ liệu OULAD thật, cần khẳng định lại số bản ghi *mồ côi* bằng 0 trước khi bàn
giao đầu vào cho việc hợp nhất *master table* (STT 4).

# Tài liệu tham khảo

[1] M. Adnan và cộng sự, "Predicting at-Risk Students at Different Percentages of Course Length
for Early Intervention Using Machine Learning Models," *IEEE Access*, vol. 9, pp. 7519–7539, 2021.

[2] J. Kuzilek, M. Hlosta, và Z. Zdrahal, "Open University Learning Analytics dataset,"
*Scientific Data*, vol. 4, art. no. 170171, 2017.

[3] A. Rule và cộng sự, "Ten simple rules for writing and sharing computational analyses in
Jupyter notebooks," *PLoS Comput. Biol.*, vol. 15, no. 7, e1007007, 2019.